# 三种零售案例 × 五类预测模型

读取已有三套 CSV，分别运行 Prophet、LightGBM、XGBoost、ARIMA 与 Temporal Fusion Transformer（TFT）。每个案例各自训练五类学习模型，并比较每周 Seasonal naive 基线；包括 15 个最终学习模型和 60 个历史起点学习模型。

运行：安装 `pyproject.toml` → 在本目录启动 Jupyter → Restart Kernel and Run All。下载包已含 `data/teaching/`。重新生成数据可先运行 `prophet-retail-data-generator.ipynb`。

默认在 CPU 上运行一个小型 TFT，包含真实训练与完整 28 天预测。训练时长受设备影响；参数均在下面集中配置。本例比较具体数据、特征与配置，不预设三个案例或五类模型的排名。

更新：增加训练历史内的 4 个代表性滚动起点、每起点重新训练的 28 天回测与 Seasonal naive 基线。最终 28 天独立测试仍保留。

## 1. 评价协议与信息边界

最后 28 天为独立测试期。更早的最后 28 天作为内部验证，LightGBM/XGBoost 用它选择树的轮数，TFT 用它选择训练 epoch；选定后重训完整训练历史。Prophet 和 ARIMA 使用固定教学配置，本次没有做等预算调参。

训练指标使用训练期末 5 个不重叠的 28 天窗口。模型参数已在完整训练历史上拟合，因而这些窗口属于训练内回看（in-sample hindcast），不构成独立回测。ARIMA 和 TFT 每个窗口仅读取起点及之前的历史状态/销量；树和 Prophet 按目标日期特征直接计算。五类模型在相同日期上评价，测试期均只从同一个最终起点预测一次。

共同可用信息是日期、VIC 假日和各场景预先已知的业务计划。TFT 另外使用过去 84 天销量，ARIMA 使用历史误差/状态；树模型本次不加销量滞后。信息使用方式存在差异，结果不能被解释为仅由算法名称造成。

训练及评价排除缺货和闭店标签。最终销售输出统一执行非负与营业日历处理；主要指标仅评价营业且未缺货日期。`conditional_mean`、`latent_demand` 只用于绘图诊断，始终不进入任何特征。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import copy, hashlib, json, logging, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.linalg import qr
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from prophet import Prophet
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback, EarlyStopping
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import RMSE

def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT")

REPO_ROOT = find_repo_root()
SEED = 42
HORIZON = 28
TRAIN_BLOCKS = 5
ENCODER_LENGTH = 84
TFT_MAX_EPOCHS = 25
TFT_BATCH_SIZE = 128
TFT_HIDDEN_SIZE = 16
CPU_THREADS = 2
DATA_DIR = REPO_ROOT / "data" / "teaching"
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '-five-models'
OUTPUT_DIR = REPO_ROOT / "runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
THREAD_LIMITER = threadpool_limits(limits=CPU_THREADS)
torch.set_num_threads(CPU_THREADS)
np.random.seed(SEED)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
PACKAGES = ["numpy", "pandas", "prophet", "holidays", "lightgbm", "xgboost",
            "statsmodels", "torch", "lightning", "pytorch-forecasting"]
VERSIONS = {name: version(name) for name in PACKAGES}
display(VERSIONS)
print("TFT: CPU, max epochs", TFT_MAX_EPOCHS, "encoder", ENCODER_LENGTH, "horizon", HORIZON)

def progress(message):
    print(message, flush=True)
    with (OUTPUT_DIR / "progress.log").open("a", encoding="utf-8") as handle:
        handle.write(message + "\n")
(OUTPUT_DIR / "progress.log").write_text("", encoding="utf-8")


## 2. 读取三套数据与共同时间划分

商超输入为折扣；红酒输入为成箱促销、12 月窗口和派对事件；庭院输入为 DIY 长周末与清仓。全部事件定义来自原生成器，不根据测试成绩改变。

缺货或闭店标签在模型专用副本中设为缺失，原始 CSV 继续保存销售记录。所有案例采用相同截止日。

In [ ]:
SCENARIOS = {
    "grocery": {"features": ["discount"], "label": "Grocery"},
    "wine": {"features": ["promo_case_buy", "is_december_peak", "is_party_event"], "label": "Wine"},
    "garden": {"features": ["is_long_weekend_diy", "is_end_of_autumn_clearance"], "label": "Garden"},
}
datasets = {}
for name, config in SCENARIOS.items():
    path = DATA_DIR / f"{name}_scenario_data.csv"
    if not path.is_file():
        raise FileNotFoundError(f"找不到 {path.resolve()}；请先运行数据生成 Notebook")
    frame = pd.read_csv(path, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
    required = {"ds", "y", "is_open", "stockout", *config["features"]}
    if not required.issubset(frame): raise ValueError(f"{name} 缺少字段 {required - set(frame)}")
    assert frame.ds.is_unique and frame.ds.notna().all()
    assert frame.ds.diff().dropna().eq(pd.Timedelta(days=1)).all()
    assert frame.y.notna().all() and frame.y.ge(0).all()
    for flag in ["is_open", "stockout"]:
        assert pd.api.types.is_bool_dtype(frame[flag]), f"{name}: {flag} 需要布尔字段"
    assert frame.loc[~frame.is_open, "y"].eq(0).all()
    assert not (frame.stockout & ~frame.is_open).any()
    assert np.isfinite(frame[config["features"]].to_numpy()).all()
    frame["time_idx"] = np.arange(len(frame))
    datasets[name] = frame
calendar_path = DATA_DIR / "vic_holidays.csv"
vic_holidays = pd.read_csv(calendar_path, parse_dates=["ds"])
assert {"ds", "holiday", "lower_window", "upper_window"}.issubset(vic_holidays)
reference_dates = datasets["grocery"].ds
assert all(frame.ds.equals(reference_dates) for frame in datasets.values())
TRAIN_END = len(reference_dates) - HORIZON - 1
INNER_END = TRAIN_END - HORIZON
cutoff = reference_dates.iloc[TRAIN_END]
assert TRAIN_END > ENCODER_LENGTH + HORIZON + HORIZON * TRAIN_BLOCKS
INPUT_HASHES = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in [calendar_path] + [DATA_DIR / f"{n}_scenario_data.csv" for n in SCENARIOS]}
display(pd.DataFrame([{"scenario": n, "days": len(f), "train_end": cutoff,
                      "validation_start": f.ds.iloc[INNER_END+1], "test_start": f.ds.iloc[TRAIN_END+1],
                      "closed_days": int((~f.is_open).sum()), "stockout_days": int(f.stockout.sum())}
                     for n,f in datasets.items()]))

def valid_labels(frame):
    return frame.is_open & ~frame.stockout & frame.y.notna()

def masked_y(frame):
    return frame.y.where(valid_labels(frame)).astype(float)

def sales_prediction(frame, raw):
    raw = np.asarray(raw, dtype=float).reshape(-1)
    assert len(raw) == len(frame) and np.isfinite(raw).all()
    return np.where(frame.is_open.to_numpy(), np.maximum(raw, 0), 0)

def metrics(frame, predicted):
    mask = valid_labels(frame).to_numpy()
    a = frame.y.to_numpy(float)[mask]; p = np.asarray(predicted)[mask]
    assert len(a) > 0 and a.sum() > 0 and np.isfinite(p).all()
    return {"MAE": float(np.abs(p-a).mean()), "RMSE": float(np.sqrt(np.mean((p-a)**2))),
            "WAPE": float(np.abs(p-a).sum()/a.sum()), "Bias": float((p-a).sum()/a.sum()),
            "n": len(a)}

## 3. 日历与业务特征

树、ARIMA 和 TFT 接收同一定义的数值日历与场景业务特征；Prophet 使用原生周期/假日模块与相同业务变量。年、周特征在所有日期采用固定 Unix 相位。

ARIMA 的连续时间索引用于动态回归趋势，傅里叶项表达周期，误差结构采用 ARIMA(1,0,1)，实现类为 statsmodels `SARIMAX`。这里不使用额外季节 AR 阶数；这是一种含外生输入的 ARIMA 比较，不是无业务输入的单变量基线。

In [ ]:
holiday_names = sorted(vic_holidays.holiday.unique())
holiday_dates = {name: pd.DatetimeIndex(vic_holidays.loc[vic_holidays.holiday.eq(name), "ds"])
                 for name in holiday_names}

def make_features(frame, scenario):
    dates = frame.ds
    days = (dates - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    X = pd.DataFrame(index=frame.index)
    X["time_years"] = (dates - pd.Timestamp("2023-01-01")) / pd.Timedelta(days=365.25)
    for label, period, order in [("weekly",7,3), ("yearly",365.25,5)]:
        for n in range(1, order+1):
            angle = 2*np.pi*n*days/period
            X[f"{label}_sin_{n}"] = np.sin(angle)
            X[f"{label}_cos_{n}"] = np.cos(angle)
    for i,name in enumerate(holiday_names):
        X[f"holiday_{i:02d}"] = dates.isin(holiday_dates[name]).astype(float)
    for col in SCENARIOS[scenario]["features"]: X[col] = frame[col].astype(float)
    assert not {"y", "latent_demand", "conditional_mean", "stockout", "is_open"}.intersection(X)
    assert np.isfinite(X.to_numpy()).all()
    return X.astype(float)

feature_names = {name: list(make_features(frame,name)) for name,frame in datasets.items()}
print({name:len(cols) for name,cols in feature_names.items()})
display(make_features(datasets["grocery"], "grocery").head())

## 4. Prophet、LightGBM、XGBoost 与 ARIMA

树模型在内部 28 天验证窗口早停，再按选出的树数量重训全部有效历史。Prophet 配置为线性趋势、周三阶、年五阶及加法回归。ARIMA 保留日历时间轴，对缺失标签采用状态空间缺失观测处理。

ARIMA 的标准化和线性独立列选择只拟合训练历史；移除全零或线性相关列避免数值退化。收敛状态另行保存，迭代未收敛时应先排查，不应仅凭指标选型。

In [ ]:
def fit_prophet(frame, scenario):
    model = Prophet(growth="linear", weekly_seasonality=False, yearly_seasonality=False,
                    daily_seasonality=False, seasonality_mode="additive", holidays=vic_holidays,
                    holidays_mode="additive", changepoint_prior_scale=0.05,
                    seasonality_prior_scale=1.0, holidays_prior_scale=1.0, uncertainty_samples=0)
    model.add_seasonality("weekly", period=7, fourier_order=3)
    model.add_seasonality("yearly", period=365.25, fourier_order=5)
    cols = SCENARIOS[scenario]["features"]
    for col in cols:model.add_regressor(col, standardize=False, prior_scale=1.0)
    training = frame[["ds","y",*cols]].copy(); training["y"] = masked_y(frame)
    model.fit(training, seed=SEED)
    return model

def fit_tree(frame, scenario, kind, inner_end=None):
    if inner_end is None: inner_end=len(frame)-HORIZON-1
    core = frame.iloc[:inner_end+1].copy(); validation = frame.iloc[inner_end+1:].copy()
    mc = valid_labels(core); mv = valid_labels(validation)
    Xc = make_features(core,scenario).loc[mc]; yc=core.loc[mc,"y"]
    Xv = make_features(validation,scenario).loc[mv]; yv=validation.loc[mv,"y"]
    if kind == "LightGBM":
        params = dict(learning_rate=.04,num_leaves=15,min_child_samples=10,reg_lambda=1.0,
                      random_state=SEED,n_jobs=CPU_THREADS,verbosity=-1)
        candidate=LGBMRegressor(n_estimators=600,**params)
        candidate.fit(Xc,yc,eval_set=[(Xv,yv)],eval_metric="mae",
                      callbacks=[early_stopping(40,verbose=False),log_evaluation(0)])
        rounds=int(candidate.best_iteration_ or 600)
        final=LGBMRegressor(n_estimators=rounds,**params)
    else:
        params=dict(objective="reg:squarederror",learning_rate=.04,max_depth=3,min_child_weight=5,
                    reg_lambda=1.0,subsample=1.0,colsample_bytree=1.0,tree_method="hist",
                    random_state=SEED,n_jobs=CPU_THREADS,eval_metric="mae")
        candidate=XGBRegressor(n_estimators=600,early_stopping_rounds=40,**params)
        candidate.fit(Xc,yc,eval_set=[(Xv,yv)],verbose=False)
        rounds=int(candidate.best_iteration+1)
        final=XGBRegressor(n_estimators=rounds,**params)
    mf=valid_labels(frame)
    final.fit(make_features(frame,scenario).loc[mf],frame.loc[mf,"y"])
    return final,{"selected_rounds":rounds,"validation_MAE":float(np.abs(candidate.predict(Xv)-yv).mean())}

def fit_arima(frame, scenario):
    X = make_features(frame,scenario)
    nonconstant = X.columns[X.loc[valid_labels(frame)].std() > 1e-8]
    scaler = StandardScaler().fit(X.loc[valid_labels(frame),nonconstant])
    scaled = scaler.transform(X[nonconstant])
    # Include the intercept when checking rank, since the model fits it as a constant trend.
    _, R, pivots = qr(np.column_stack([np.ones(len(frame)),scaled])[valid_labels(frame)],pivoting=True,mode="economic")
    rank = int((np.abs(np.diag(R)) > 1e-8).sum())
    selected = sorted(int(j-1) for j in pivots[:rank] if j>0)
    scaled = scaled[:,selected]
    y=masked_y(frame).to_numpy()
    fitted=SARIMAX(y,exog=scaled,order=(1,0,1),trend="c",
                   enforce_stationarity=True,enforce_invertibility=True).fit(disp=False,maxiter=150)
    return {"result":fitted,"scaler":scaler,"nonconstant":list(nonconstant),
            "selected":selected,"y":y,"exog":scaled}, {
                "order":"(1,0,1)","converged":bool(fitted.mle_retvals.get("converged",False)),
                "exog_columns":len(selected)}

def arima_features(bundle, frame, scenario):
    return bundle["scaler"].transform(make_features(frame,scenario)[bundle["nonconstant"]])[:,bundle["selected"]]

def classical_forecast(model, kind, frame, scenario, origin=None):
    if kind=="Prophet":
        return model.predict(frame[["ds",*SCENARIOS[scenario]["features"]]])["yhat"].to_numpy()
    if kind in ["LightGBM","XGBoost"]: return model.predict(make_features(frame,scenario))
    fitted=model["result"]
    if origin is not None: # Same fitted parameters; historical state ends at this block's origin.
        fitted=fitted.apply(model["y"][:origin+1],exog=model["exog"][:origin+1],refit=False)
    return np.asarray(fitted.get_forecast(len(frame),exog=arima_features(model,frame,scenario)).predicted_mean)
def seasonal_naive(history, future):
    # Freeze one latest valid observation per weekday at the forecast origin.
    # Missing/closed/stockout days may require looking back into different weeks.
    usable=history.loc[valid_labels(history), ["ds","y"]].copy()
    usable["weekday"]=usable.ds.dt.dayofweek
    latest=usable.groupby("weekday",sort=False).tail(1).set_index("weekday").y
    if len(latest)!=7: raise ValueError("基线需要各星期位置至少一次有效历史观测")
    return future.ds.dt.dayofweek.map(latest).to_numpy(float)


## 5. TFT：真实多步训练与无未来销量泄漏的推理

采用 PyTorch Forecasting 的 `TemporalFusionTransformer`，每个案例独立训练，84 天编码器窗口，28 天解码器窗口。日历与业务计划属于已知未来输入；销量只属于历史未知变量，解码器不读取未来销量。

TFT 数据集要求目标有限。本例把无效历史标签用截止当前日的最近有效销量前向填充，并以零权重排除这些日期的解码器损失；首部缺失只用训练有效标签均值补齐。目标均值与标准差从训练有效标签计算。历史插补仍是一个工程假设，不能恢复真实缺货需求。

预测时未来目标统一置为占位符，权重为零；之后改成另一个极端占位值并断言预测不变。验证数据不进入目标缩放与特征标准化的拟合。训练最终模型使用内部验证选出的 epoch 数，最后测试期不参与选择。

In [ ]:
class EpochChoice(Callback):
    def __init__(self,label):
        self.label=label;self.best_loss=float("inf");self.best_epoch=1
    def on_validation_epoch_end(self,trainer,model):
        if trainer.sanity_checking:return
        value=float(trainer.callback_metrics["val_loss"])
        if value < self.best_loss:self.best_loss=value;self.best_epoch=trainer.current_epoch+1
        if (trainer.current_epoch+1)%5==0:
            progress(f"{self.label}: epoch {trainer.current_epoch+1}, val_loss={value:.4f}")

class TrainProgress(Callback):
    def __init__(self,label):self.label=label
    def on_train_epoch_end(self,trainer,model):
        if (trainer.current_epoch+1)%5==0:
            progress(f"{self.label}: epoch {trainer.current_epoch+1}")

def tft_frame(frame, scenario, mean, std):
    out=make_features(frame,scenario).reset_index(drop=True)
    out["time_idx"]=frame.time_idx.to_numpy(dtype=int)
    out["series"]=scenario
    y=masked_y(frame).ffill().fillna(mean).to_numpy()
    out["target_z"]=(y-mean)/std
    out["label_weight"]=valid_labels(frame).astype(float).to_numpy()
    return out

def training_dataset(frame,scenario):
    usable=frame.loc[valid_labels(frame),"y"]
    mean=float(usable.mean());std=max(float(usable.std()),1.0)
    prepared=tft_frame(frame,scenario,mean,std)
    dataset=TimeSeriesDataSet(prepared,time_idx="time_idx",target="target_z",group_ids=["series"],
              weight="label_weight",min_encoder_length=ENCODER_LENGTH,max_encoder_length=ENCODER_LENGTH,
              min_prediction_length=HORIZON,max_prediction_length=HORIZON,
              static_categoricals=["series"],time_varying_known_reals=feature_names[scenario],
              time_varying_unknown_reals=["target_z"],target_normalizer=None,
              add_relative_time_idx=True,randomize_length=False)
    return dataset,mean,std

def tft_inference_dataset(bundle, full_frame, scenario, origin, placeholder=0.0):
    assert origin+HORIZON < len(full_frame)
    visible=full_frame.iloc[:origin+HORIZON+1].copy()
    # Hide all target labels and diagnostic truth after the origin before preprocessing.
    visible.loc[visible.time_idx > origin,"y"]=np.nan
    prepared=tft_frame(visible,scenario,bundle["mean"],bundle["std"])
    decoder=prepared.time_idx > origin
    prepared.loc[decoder,"target_z"]=placeholder
    prepared.loc[decoder,"label_weight"]=0.0
    return TimeSeriesDataSet.from_dataset(bundle["dataset"],prepared,predict=True,stop_randomization=True)

def new_tft(dataset):
    return TemporalFusionTransformer.from_dataset(dataset,learning_rate=.005,
               hidden_size=TFT_HIDDEN_SIZE,attention_head_size=2,hidden_continuous_size=8,
               lstm_layers=1,dropout=.1,output_size=1,loss=RMSE(),
               reduce_on_plateau_patience=5,log_interval=-1)

def trainer(epochs,callbacks=()):
    return pl.Trainer(accelerator="cpu",devices=1,max_epochs=epochs,logger=False,
               enable_checkpointing=False,enable_progress_bar=False,enable_model_summary=False,
               callbacks=list(callbacks),gradient_clip_val=.1,num_sanity_val_steps=0,
               deterministic=True,default_root_dir=str(OUTPUT_DIR / "lightning"))

def fit_tft(full_frame,scenario,train_end=None,checkpoint_tag=None):
    if train_end is None: train_end=TRAIN_END
    inner_end=train_end-HORIZON
    if checkpoint_tag is None: checkpoint_tag=scenario
    history=full_frame.iloc[:train_end+1].copy()
    core=history.iloc[:inner_end+1].copy()
    core_dataset,mean,std=training_dataset(core,scenario)
    selection_std=std
    # Validation labels are used for the loss only; they are not decoder input variables.
    validation_prepared=tft_frame(history,scenario,mean,std)
    validation_dataset=TimeSeriesDataSet.from_dataset(core_dataset,validation_prepared,
                                                      predict=True,stop_randomization=True)
    pl.seed_everything(SEED,workers=True)
    selection=new_tft(core_dataset)
    choice=EpochChoice(scenario+" selection")
    train_loader=core_dataset.to_dataloader(train=True,batch_size=TFT_BATCH_SIZE,num_workers=0)
    validation_loader=validation_dataset.to_dataloader(train=False,batch_size=1,num_workers=0)
    selection_trainer=trainer(TFT_MAX_EPOCHS,[choice,EarlyStopping(monitor="val_loss",patience=6,mode="min")])
    selection_trainer.fit(selection,train_dataloaders=train_loader,val_dataloaders=validation_loader)
    epochs=choice.best_epoch
    dataset,mean,std=training_dataset(history,scenario)
    pl.seed_everything(SEED,workers=True)
    final=new_tft(dataset)
    final_trainer=trainer(epochs,[TrainProgress(scenario+" final")])
    final_trainer.fit(final,train_dataloaders=dataset.to_dataloader(
         train=True,batch_size=TFT_BATCH_SIZE,num_workers=0))
    final.eval()
    final_trainer.save_checkpoint(OUTPUT_DIR / f"{checkpoint_tag}_tft.ckpt")
    return {"model":final,"dataset":dataset,"mean":mean,"std":std}, {
       "selected_epochs":epochs,"validation_RMSE_original":float(choice.best_loss*selection_std),
       "encoder_length":ENCODER_LENGTH,"hidden_size":TFT_HIDDEN_SIZE,
       "target_mean":mean,"target_std":std,"validation_target_std":selection_std}

def predict_tft(bundle,full_frame,scenario,origin,placeholder=0.0):
    dataset=tft_inference_dataset(bundle,full_frame,scenario,origin,placeholder)
    loader=dataset.to_dataloader(train=False,batch_size=1,num_workers=0)
    raw=bundle["model"].predict(loader,mode="prediction",trainer_kwargs={
         "accelerator":"cpu","devices":1,"logger":False,"enable_progress_bar":False,
         "enable_model_summary":False,"enable_checkpointing":False})
    if hasattr(raw,"output"):raw=raw.output
    values=raw.detach().cpu().numpy().reshape(-1)*bundle["std"]+bundle["mean"]
    assert len(values)==HORIZON
    return values

## 6. 运行三个案例 × 五类模型

此单元会真实训练全部模型，输出进度、耗时、选择的轮数/epoch 以及 ARIMA 收敛状态。训练回看窗口相邻而不重叠；完整训练参数已经看到这些日期，不能把训练成绩当作预测泛化证据。

未来标签替换测试用于检查 TFT 的输入路径；它只检查推理不读未来销量，不替代整个训练流程的信息审计。

In [ ]:
model_names=["Prophet","LightGBM","XGBoost","ARIMA","TFT","Seasonal naive"]
trained_models={};fit_records=[];metric_rows=[];prediction_frames=[]
train_starts=list(range(TRAIN_END-HORIZON*TRAIN_BLOCKS+1,TRAIN_END+1,HORIZON))
assert len(train_starts)==TRAIN_BLOCKS
for scenario,frame in datasets.items():
    history=frame.iloc[:TRAIN_END+1].copy();test=frame.iloc[TRAIN_END+1:].copy()
    for kind in model_names:
        progress(f"START {scenario} / {kind}")
        started=time.perf_counter(); details={}
        if kind=="Prophet":fitted=fit_prophet(history,scenario)
        elif kind in ["LightGBM","XGBoost"]:fitted,details=fit_tree(history,scenario,kind)
        elif kind=="ARIMA":fitted,details=fit_arima(history,scenario)
        elif kind=="TFT":fitted,details=fit_tft(frame,scenario)
        else:fitted=history.copy()
        seconds=time.perf_counter()-started
        trained_models[(scenario,kind)]=fitted
        fit_records.append({"scenario":scenario,"model":kind,"fit_seconds":seconds,**details})
        # Check test inference under two arbitrary future target placeholders.
        if kind=="TFT":
            raw_test=predict_tft(fitted,frame,scenario,TRAIN_END)
            changed=predict_tft(fitted,frame,scenario,TRAIN_END,placeholder=12345.0)
            np.testing.assert_allclose(raw_test,changed,rtol=1e-6,atol=1e-5)
        elif kind=="Seasonal naive":raw_test=seasonal_naive(history,test)
        else:raw_test=classical_forecast(fitted,kind,test,scenario)
        blocks=[]
        for start in train_starts:
            block=frame.iloc[start:start+HORIZON].copy();origin=start-1
            raw=seasonal_naive(frame.iloc[:origin+1],block) if kind=="Seasonal naive" else predict_tft(fitted,frame,scenario,origin) if kind=="TFT" else classical_forecast(
                  fitted,kind,block,scenario,origin=origin if kind=="ARIMA" else None)
            block["raw_prediction"]=raw;block["prediction"]=sales_prediction(block,raw)
            block["forecast_origin"]=frame.ds.iloc[origin];blocks.append(block)
        train_eval=pd.concat(blocks,ignore_index=True)
        test_eval=test.copy();test_eval["raw_prediction"]=raw_test
        test_eval["prediction"]=sales_prediction(test,raw_test);test_eval["forecast_origin"]=cutoff
        for split,evaluated in [("train_hindcast",train_eval),("test",test_eval)]:
            metric_rows.append({"scenario":scenario,"model":kind,"split":split,
                                **metrics(evaluated,evaluated.prediction)})
            evaluated["scenario"]=scenario;evaluated["model"]=kind;evaluated["split"]=split
            prediction_frames.append(evaluated)
        # Save checkpoints progressively so completed fits remain reviewable.
        pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR / "metrics.csv",index=False)
        pd.DataFrame(fit_records).to_csv(OUTPUT_DIR / "fit_details.csv",index=False)
        progress(f"DONE {scenario} / {kind}: {seconds:.1f}s, test WAPE={metric_rows[-1]['WAPE']:.2%}")
all_predictions=pd.concat(prediction_frames,ignore_index=True)
all_metrics=pd.DataFrame(metric_rows);fit_details=pd.DataFrame(fit_records)
assert len(all_metrics)==3*6*2
assert set(all_metrics.model)==set(model_names)
for (scenario,split),group in all_metrics.groupby(["scenario","split"]):
    assert group.n.nunique()==1, "模型的评价日期数不一致"
all_predictions.to_csv(OUTPUT_DIR / "predictions.csv",index=False)
display(all_metrics.round(4))
display(fit_details)

## 7. 训练与测试 performance 图

每个案例生成训练回看、测试预测与训练/测试误差图。不同案例单位和规模不同，跨案例比较优先看相对误差与业务条件；WAPE 也受评价期的实际销量总量影响。

测试只有一个年末窗口，不能代表全年、更不能代表真实品牌商品的性能。训练和测试之间的差距还受季节与事件分布变化影响。

In [ ]:
COLORS=dict(zip(model_names,["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:gray"]))
for scenario,frame in datasets.items():
    fig,axes=plt.subplots(3,1,figsize=(13,12),constrained_layout=True)
    for ax,split,title in [(axes[0],"train_hindcast","Training-period hindcast (in sample)"),
                           (axes[1],"test","Independent 28-day test")]:
        group=all_predictions.loc[(all_predictions.scenario==scenario)&(all_predictions.split==split)]
        first=group.loc[group.model==model_names[0]]
        ax.plot(first.ds,first.y,"o-",color="black",linewidth=1,markersize=3,label="Observed sales")
        excluded_stockout=first.loc[first.stockout & first.is_open]
        excluded_closed=first.loc[~first.is_open]
        if len(excluded_stockout):
            ax.scatter(excluded_stockout.ds,excluded_stockout.y,color="darkorange",marker="x",
                       s=60,label="Stockout (excluded)",zorder=5)
        if len(excluded_closed):
            ax.scatter(excluded_closed.ds,excluded_closed.y,color="grey",marker="s",
                       s=35,label="Closed (excluded)",zorder=5)
        for kind in model_names:
            values=group.loc[group.model==kind]
            ax.plot(values.ds,values.prediction,color=COLORS[kind],alpha=.85,label=kind)
        ax.set(title=f"{SCENARIOS[scenario]['label']}: {title}",ylabel="Units");ax.legend(ncol=3)
        ax.tick_params(axis="x",rotation=20)
    values=all_metrics.loc[all_metrics.scenario==scenario].pivot(index="model",columns="split",values="WAPE").reindex(model_names)
    x=np.arange(len(model_names))
    axes[2].bar(x-.18,values.train_hindcast*100,width=.36,label="Train hindcast")
    axes[2].bar(x+.18,values.test*100,width=.36,label="Test")
    axes[2].set(xticks=x,xticklabels=model_names,ylabel="WAPE (%)",title="Model metrics on common evaluation dates")
    axes[2].legend()
    fig.savefig(OUTPUT_DIR / f"{scenario}_train_test.png",dpi=150,bbox_inches="tight")
    plt.show();plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(15,4),constrained_layout=True)
for ax,metric in zip(axes,["WAPE","MAE","RMSE"]):
    table=all_metrics.loc[all_metrics.split=="test"].pivot(index="scenario",columns="model",values=metric)
    table=table.reindex(index=list(SCENARIOS),columns=model_names)
    shown=table.to_numpy()*100 if metric=="WAPE" else table.to_numpy()
    image=ax.imshow(shown,aspect="auto",cmap="YlOrRd")
    ax.set(xticks=np.arange(len(model_names)),xticklabels=model_names,yticks=np.arange(3),yticklabels=list(SCENARIOS),
           title="Test "+metric+(" (%)" if metric=="WAPE" else " (units)"))
    ax.tick_params(axis="x",rotation=45)
    for i in range(3):
        for j in range(len(model_names)):ax.text(j,i,f"{shown[i,j]:.2f}",ha="center",va="center",fontsize=8)
    fig.colorbar(image,ax=ax,shrink=.8)
fig.savefig(OUTPUT_DIR / "test_metrics_heatmap.png",dpi=150,bbox_inches="tight")
plt.show();plt.close(fig)

## 8. 测试误差与训练成本

累计偏差可提示补货总体偏高或偏低；日级残差仅代表本窗口。耗时包含内部验证选择与最终重训，Prophet/ARIMA 未做参数搜索，所以不能当作等预算效率基准。

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    group=all_predictions.loc[(all_predictions.scenario==scenario)&(all_predictions.split=="test")]
    for kind in model_names:
        values=group.loc[(group.model==kind)&group.is_open&~group.stockout]
        ax.plot(values.ds,values.prediction-values.y,label=kind,color=COLORS[kind])
    ax.axhline(0,color="black",linewidth=.8);ax.set(title=scenario,ylabel="Prediction - observed")
    ax.tick_params(axis="x",rotation=35)
axes[-1].legend(fontsize=8)
fig.savefig(OUTPUT_DIR / "test_errors.png",dpi=150,bbox_inches="tight");plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4),constrained_layout=True)
fit_details.pivot(index="scenario",columns="model",values="fit_seconds").reindex(columns=model_names).plot.bar(ax=ax)
ax.set(ylabel="Seconds (selection + refit)",title="Measured CPU training cost")
fig.savefig(OUTPUT_DIR / "fit_times.png",dpi=150,bbox_inches="tight");plt.show();plt.close(fig)

### 回测前的结果解释边界

本实验已实际运行五类模型，但 TFT 的小型局部训练、ARIMA 阶数、树配置和年末测试窗口均属于特定设定。可以在训练历史中增加代表性窗口、合理调参预算及更多随机种子，再评价稳定性；保留最终测试期独立。

红酒使用计数生成机制，当前五类模型仍使用连续点预测与统一非负处理；没有把 TFT 改为泊松/负二项输出。生产部署需要根据需求分布、输入质量和补货成本进一步验证。

训练内成绩越好不一定表示未来预测越好，跨模型排名需同时注明历史输入差异与信息条件。

## 9. 训练历史内的滚动起点回测（rolling-origin backtesting）

预先根据生成公式与业务日历指定四个窗口，选择过程不读取预测误差或最终测试标签。每个窗口向前使用全部可用历史，重新拟合六类模型，预测未来 28 天。树模型和 TFT 在该起点之前再留 28 天做内部验证；缩放、标准化、缺失处理和 ARIMA 特征选择均只拟合当时的训练历史。

商超：普通期 2 月、秋季旺季 4 月、促销观察期 5 月、低谷 10 月。红酒：普通期 3 月、冬季旺季 7 月、成箱促销观察期 8 月、夏季低谷 2 月。庭院：普通期 2 月、秋季旺季 4 月、清仓 5 月、低谷 10 月。

“普通”“旺季”“淡季”是教学机制下的预期标签，不根据实际销量分位数认定。商超和红酒促销全年循环，促销观察期仍混合促销与非促销日；另给所有回测的业务条件切片，条件标签可以重叠，切片不构成互斥数据集。

Seasonal naive 在每个起点保存各星期位置最近一次有效销量，并重复用于未来四周；无效日期向更早周回溯。它不能利用未来折扣、节庆或趋势。所有模型采用相同营业日处理和评价日期。

窗口均在最终训练截止日前结束，且同案例的窗口互不重叠。四个代表性起点用于教学诊断，尚不能覆盖全年全部需求机制；这里固定结构，不用回测结果反复调整配置。最后 28 天测试结果沿用前节协议，并单独展示。


基线的数学定义为：

$$\ell(d;T)=\max\{s\leq T:\operatorname{weekday}(s)=\operatorname{weekday}(d),\ s\text{ 为有效观测日}\},\qquad \hat y(d\mid T)=y_{\ell(d;T)}.$$

变量定义：$T$ 是预测起点日期，$d$ 是未来日期，$s$ 是历史日期，$\ell(d;T)$ 是起点之前与未来日同星期位置的最近有效日期；$y_s$ 是历史销量，$\hat y(d\mid T)$ 是起点 $T$ 对日期 $d$ 的预测。$\max$ 表示取满足条件的最大日期，$\operatorname{weekday}$ 返回星期位置。无缺失时，它就是重复最近一周的七个值；本例向更早周回溯无效标签 [7]。


In [ ]:
WINDOW_STARTS = {
 "grocery": [("ordinary","2025-02-03"),("peak","2025-04-07"),("promotion","2025-05-19"),("low","2025-10-06")],
 "wine": [("ordinary","2025-03-03"),("peak","2025-07-07"),("promotion","2025-08-04"),("low","2025-02-03")],
 "garden": [("ordinary","2025-02-03"),("peak","2025-04-07"),("promotion","2025-05-19"),("low","2025-10-06")],
}
window_rows=[]
for scenario,windows in WINDOW_STARTS.items():
    frame=datasets[scenario]
    seen=set()
    for regime,start_date in windows:
        start=int(frame.index[frame.ds.eq(pd.Timestamp(start_date))][0]);end=start+HORIZON-1
        assert end<=TRAIN_END and start>ENCODER_LENGTH+2*HORIZON
        dates=set(range(start,end+1));assert not seen.intersection(dates);seen.update(dates)
        window_rows.append({"scenario":scenario,"regime":regime,"origin":frame.ds.iloc[start-1],
                            "start":frame.ds.iloc[start],"end":frame.ds.iloc[end],"start_idx":start})
windows_table=pd.DataFrame(window_rows)
display(windows_table.drop(columns="start_idx"))

cv_rows=[];cv_predictions=[];cv_fits=[]
for window in window_rows:
    scenario=window["scenario"];regime=window["regime"];start=window["start_idx"]
    origin=start-1;frame=datasets[scenario];history=frame.iloc[:start].copy()
    # The fitting input itself ends at the origin. No future target rows are supplied to fit_tft.
    target=frame.iloc[start:start+HORIZON].copy()
    assert history.ds.max()<target.ds.min() and target.ds.max()<=cutoff
    for kind in model_names:
        progress(f"CV START {scenario}/{regime}/{kind}: {history.ds.max().date()}")
        t0=time.perf_counter();details={}
        if kind=="Prophet":fitted=fit_prophet(history,scenario)
        elif kind in ["LightGBM","XGBoost"]:fitted,details=fit_tree(history,scenario,kind)
        elif kind=="ARIMA":fitted,details=fit_arima(history,scenario)
        elif kind=="TFT":fitted,details=fit_tft(history,scenario,train_end=origin,checkpoint_tag=f"{scenario}_{regime}")
        else:fitted=history
        seconds=time.perf_counter()-t0
        if kind=="TFT":
            raw=predict_tft(fitted,frame,scenario,origin)
            changed=predict_tft(fitted,frame,scenario,origin,placeholder=12345)
            np.testing.assert_allclose(raw,changed,rtol=1e-6,atol=1e-5)
        elif kind=="Seasonal naive":raw=seasonal_naive(history,target)
        else:raw=classical_forecast(fitted,kind,target,scenario)
        result=target.copy();result["prediction"]=sales_prediction(target,raw);result["raw_prediction"]=raw
        result["scenario"]=scenario;result["regime"]=regime;result["model"]=kind
        result["forecast_origin"]=window["origin"];result["horizon_day"]=np.arange(1,HORIZON+1)
        cv_predictions.append(result)
        cv_rows.append({"scenario":scenario,"regime":regime,"model":kind,"origin":window["origin"],
                        **metrics(result,result.prediction)})
        cv_fits.append({"scenario":scenario,"regime":regime,"model":kind,"fit_seconds":seconds,**details})
        pd.DataFrame(cv_rows).to_csv(OUTPUT_DIR/"rolling_metrics.csv",index=False)
        pd.DataFrame(cv_fits).to_csv(OUTPUT_DIR/"rolling_fit_details.csv",index=False)
        pd.concat(cv_predictions,ignore_index=True).to_csv(OUTPUT_DIR/"rolling_predictions.csv",index=False)
        progress(f"CV DONE {scenario}/{regime}/{kind}: {seconds:.1f}s, WAPE={cv_rows[-1]['WAPE']:.2%}")
        del fitted
cv_metrics=pd.DataFrame(cv_rows);cv_predictions=pd.concat(cv_predictions,ignore_index=True)
cv_fit_details=pd.DataFrame(cv_fits)
assert len(cv_metrics)==3*4*6
for _,group in cv_predictions.groupby(["scenario","regime"]):
    sets=[set(g.loc[valid_labels(g),"ds"]) for _,g in group.groupby("model")]
    assert all(d==sets[0] for d in sets)
assert cv_predictions.ds.max()<=cutoff
assert cv_fit_details.loc[cv_fit_details.model.eq("ARIMA"),"converged"].all()
display(cv_metrics.round(4))


## 10. 相对基线的收益、业务切片与预测步长

汇总 WAPE 使用全部回测有效日期的绝对误差总和除以销量总和；“窗口平均 WAPE”给四个窗口相同权重。两者回答不同问题。基线改善率 = 1 − 模型 WAPE / 基线 WAPE，正值表示降低误差，负值表示低于基线。它不是统计显著性检验。

业务切片分别评价促销、非促销、假日和普通营业日；普通日同时排除促销和假日。额外给出精确预测步长的误差，但每个案例/步长只有至多四个有效样本，宜看分段趋势而不强调单日波动。

训练内回看、历史滚动验证、最终测试分别报告，避免混淆。最终测试只能支持当前配置和这一个窗口；后续若依照已查看的测试结果改模型，应建立新的未查看测试期。


In [ ]:
summary_rows=[]
for (scenario,kind),g in cv_predictions.groupby(["scenario","model"]):
    summary_rows.append({"scenario":scenario,"model":kind,**metrics(g,g.prediction),
       "mean_window_WAPE":float(cv_metrics.loc[(cv_metrics.scenario==scenario)&(cv_metrics.model==kind),"WAPE"].mean()),
       "total_fit_seconds":float(cv_fit_details.loc[(cv_fit_details.scenario==scenario)&(cv_fit_details.model==kind),"fit_seconds"].sum())})
rolling_summary=pd.DataFrame(summary_rows)
base=rolling_summary.loc[rolling_summary.model.eq("Seasonal naive")].set_index("scenario").WAPE
rolling_summary["improvement_vs_naive"]=1-rolling_summary.WAPE/rolling_summary.scenario.map(base)

slice_rows=[];horizon_rows=[]
all_holidays=set(vic_holidays.ds)
for (scenario,kind),g in cv_predictions.groupby(["scenario","model"]):
    promo=g[{"grocery":"discount","wine":"promo_case_buy","garden":"is_end_of_autumn_clearance"}[scenario]].gt(0)
    holiday=g.ds.isin(all_holidays)
    for label,mask in [("promotion",promo),("non_promotion",~promo),("holiday",holiday),("ordinary",~promo&~holiday)]:
        selected=g.loc[mask]
        if valid_labels(selected).any():slice_rows.append({"scenario":scenario,"model":kind,"condition":label,**metrics(selected,selected.prediction)})
    for step,h in g.groupby("horizon_day"):
        if valid_labels(h).any():horizon_rows.append({"scenario":scenario,"model":kind,"horizon_day":step,**metrics(h,h.prediction)})
business_slices=pd.DataFrame(slice_rows);horizon_metrics=pd.DataFrame(horizon_rows)
for name,table in [("rolling_windows",windows_table),("rolling_summary",rolling_summary),
                   ("business_slices",business_slices),("horizon_metrics",horizon_metrics)]:
    table.to_csv(OUTPUT_DIR/f"{name}.csv",index=False)
display(rolling_summary.round(4));display(business_slices.round(4))

fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    table=cv_metrics.loc[cv_metrics.scenario.eq(scenario)].pivot(index="regime",columns="model",values="WAPE").reindex(index=["ordinary","peak","promotion","low"],columns=model_names)
    values=table.to_numpy()*100
    im=ax.imshow(values,cmap="YlOrRd",aspect="auto")
    ax.set(xticks=range(len(model_names)),xticklabels=model_names,yticks=range(4),yticklabels=table.index,title=scenario+" rolling WAPE (%)")
    ax.tick_params(axis="x",rotation=50)
    for i in range(4):
        for j in range(len(model_names)):ax.text(j,i,f"{values[i,j]:.1f}",ha="center",va="center",fontsize=8)
    fig.colorbar(im,ax=ax,shrink=.7)
fig.savefig(OUTPUT_DIR/"rolling_wape_heatmap.png",dpi=150);plt.show();plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    data=rolling_summary.loc[rolling_summary.scenario.eq(scenario)].set_index("model").reindex(model_names)
    ax.bar(range(len(data)),data.improvement_vs_naive*100,color=[COLORS[k] for k in model_names]);ax.axhline(0,color="black",lw=.8)
    ax.set(xticks=range(len(data)),xticklabels=model_names,title=scenario,ylabel="WAPE reduction vs naive (%)");ax.tick_params(axis="x",rotation=50)
fig.savefig(OUTPUT_DIR/"rolling_baseline_gain.png",dpi=150);plt.show();plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    train=all_metrics.loc[(all_metrics.scenario==scenario)&(all_metrics.split=="train_hindcast")].set_index("model").WAPE.reindex(model_names)
    cv=rolling_summary.loc[rolling_summary.scenario.eq(scenario)].set_index("model").WAPE.reindex(model_names)
    test=all_metrics.loc[(all_metrics.scenario==scenario)&(all_metrics.split=="test")].set_index("model").WAPE.reindex(model_names)
    pd.DataFrame({"Train in-sample":train*100,"Rolling validation":cv*100,"Final test":test*100}).plot.bar(ax=ax)
    ax.set(title=scenario,ylabel="WAPE (%)");ax.tick_params(axis="x",rotation=50)
fig.savefig(OUTPUT_DIR/"train_cv_test.png",dpi=150);plt.show();plt.close(fig)

for scenario in SCENARIOS:
    fig,axes=plt.subplots(4,1,figsize=(13,12),constrained_layout=True)
    for ax,regime in zip(axes,["ordinary","peak","promotion","low"]):
        group=cv_predictions.loc[(cv_predictions.scenario==scenario)&(cv_predictions.regime==regime)]
        actual=group.loc[group.model=="Seasonal naive"]
        ax.plot(actual.ds,actual.y,"o-",color="black",ms=3,label="Observed")
        for kind in model_names:
            g=group.loc[group.model==kind];ax.plot(g.ds,g.prediction,color=COLORS[kind],label=kind,alpha=.8)
        ax.set(title=f"{scenario}: {regime}",ylabel="Units");ax.tick_params(axis="x",rotation=15)
    axes[0].legend(ncol=4)
    fig.savefig(OUTPUT_DIR/f"{scenario}_rolling_forecasts.png",dpi=150);plt.show();plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    for kind in model_names:
        g=horizon_metrics.loc[(horizon_metrics.scenario==scenario)&(horizon_metrics.model==kind)]
        ax.plot(g.horizon_day,g.MAE,label=kind,color=COLORS[kind])
    ax.set(title=scenario,xlabel="Exact forecast day (<=4 observations)",ylabel="MAE")
axes[-1].legend(fontsize=8)
fig.savefig(OUTPUT_DIR/"rolling_horizon_mae.png",dpi=150);plt.show();plt.close(fig)


## 11. 导出与复现

保存原始训练/测试和新增回测结果。

In [ ]:
record={"versions":VERSIONS,"seed":SEED,"cutoff":str(cutoff.date()),"horizon":HORIZON,
        "train_evaluation":"last five 28-day blocks; fitted parameters have seen these labels",
        "test_evaluation":"single 28-day holdout, open non-stockout dates",
        "tft_max_epochs":TFT_MAX_EPOCHS,"encoder_length":ENCODER_LENGTH,
        "input_sha256":INPUT_HASHES,"all_models_executed":sorted(all_metrics.model.unique()),
        "future_target_placeholder_check":"TFT predictions equal for 0 and 12345",
        "arima_convergence":fit_details.loc[fit_details.model=="ARIMA",["scenario","converged"]].to_dict("records")}
# Save original-unit target scaling alongside the TFT checkpoints.
target_scaling = {}
for scenario, frame in datasets.items():
    historical = frame.iloc[:TRAIN_END+1]
    usable = historical.loc[valid_labels(historical), "y"]
    target_scaling[scenario] = {"mean": float(usable.mean()), "std": max(float(usable.std()), 1.0),
                                "feature_columns": feature_names[scenario]}
record["tft_target_scaling"] = target_scaling
for (scenario, split), group in all_predictions.groupby(["scenario", "split"]):
    common = None
    for model_name, values in group.groupby("model"):
        dates = set(values.loc[valid_labels(values), "ds"])
        if common is None: common = dates
        else: assert dates == common, "模型评价日期不一致"
for name,digest in INPUT_HASHES.items():
    assert hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()==digest
(OUTPUT_DIR / "comparison_manifest.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print("已导出图表、预测、指标、训练配置与输入校验和：",OUTPUT_DIR.resolve())
display(all_metrics.loc[all_metrics.split=="test"].pivot(index="scenario",columns="model",values="WAPE"))
record["rolling_origins"]=windows_table.drop(columns="start_idx").astype(str).to_dict("records")
record["rolling_protocol"]="fresh refit per origin, nested 28-day validation; 4 nonoverlapping windows per scenario; final test untouched"
record["rolling_arima_convergence"]=cv_fit_details.loc[cv_fit_details.model.eq("ARIMA"),["scenario","regime","converged"]].to_dict("records")
(OUTPUT_DIR/"comparison_manifest.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")

record["rolling_tft_target_scaling"] = {}
for row in window_rows:
    scenario=row["scenario"];history=datasets[scenario].iloc[:row["start_idx"]]
    usable=history.loc[valid_labels(history),"y"]
    record["rolling_tft_target_scaling"][scenario+"_"+row["regime"]]={
        "mean":float(usable.mean()),"std":max(float(usable.std()),1.0),"origin":str(row["origin"].date())}
(OUTPUT_DIR/"comparison_manifest.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")


## 参考

[1] Prophet. Seasonality, Holiday Effects, and Regressors. https://facebook.github.io/prophet/docs/seasonality,_holiday_effects,_and_regressors.html

[2] LightGBM. Python API. https://lightgbm.readthedocs.io/en/v4.6.0/Python-API.html

[3] XGBoost. Python API. https://xgboost.readthedocs.io/en/stable/python/python_api.html

[4] statsmodels. SARIMAX. https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html

[5] PyTorch Forecasting. Demand forecasting with the Temporal Fusion Transformer. https://pytorch-forecasting.readthedocs.io/en/v1.5.0/tutorials/stallion.html

[6] Lim et al. Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting. https://arxiv.org/abs/1912.09363

[7] Hyndman, R. J., & Athanasopoulos, G. Forecasting: Principles and Practice (3rd ed.). Some simple forecasting methods; Time series cross-validation. https://otexts.com/fpp3/simple-methods.html ; https://otexts.com/fpp3/tscv.html